# 00.3 — Vectors: norm, dot product, cosine

**Question:** how do I measure *how long* a vector is, and *how much two vectors agree*?

**Interview one-liners**
- A vector dotted with itself gives its length squared: `a · a = |a|²`, so `|a| = √(a · a)`.
- A dot product is big for **two different reasons** — the vectors are well aligned, *or* the vectors are simply long. That ambiguity is the whole story.
- Cosine similarity divides the lengths out, so it reports alignment alone and is always in `[-1, 1]` regardless of dimension. That bounded range is why search systems rank with it.
- The same number has two names: the **algebraic (component) form** `Σ aᵢbᵢ` is what the computer runs; the **geometric form** `|a||b|cos θ` is what it means.
- Matrix multiplication is a **grid of dot products** — every cell of the output is one row dotted with one column.

In [ ]:
import torch

torch.manual_seed(0)

# a is the arrow from the origin to the point (3, 4) on graph paper.
# b is a doubled  -> same direction, twice as long.
# c is b negated  -> same length as b, exact opposite direction.
a = torch.tensor([3.0, 4.0])
b = torch.tensor([6.0, 8.0])
c = torch.tensor([-6.0, -8.0])

# One axis, two elements: a single vector, not a batch and not a matrix.
print('shapes:', a.shape, b.shape, c.shape)

## Experiment 1 — the norm, two ways

On graph paper, `a = [3, 4]` is the hypotenuse of a right triangle with legs 3 and 4.
Pythagoras says its length is `√(3² + 4²) = √25 = 5`.

Notice that `3² + 4²` is *exactly* what `a · a` computes. So the claim is `|a| = √(a · a)`.

In [ ]:
print('a . a                =', torch.dot(a, a).item())        # 3*3 + 4*4
print('sqrt(a . a)          =', torch.dot(a, a).sqrt().item())  # our claim
print('torch.linalg.norm(a) =', torch.linalg.norm(a).item())    # PyTorch's built-in

### Reading the output
`a · a = 25`, and both `√(a · a)` and PyTorch's `linalg.norm` return `5.0`.

They agree, which is the point: the identity is not a coincidence of the 3-4-5 triangle,
it is the **definition** of length. `linalg.norm` is doing exactly this square-root-of-self-dot.

## Experiment 2 — dot product, then cosine

**Algebraic form** (the recipe the computer runs): multiply matching positions, add them up.

    a · b = (3×6) + (4×8) = 18 + 32 = 50

**Geometric form** (what the number means): the two lengths multiplied, times how aligned they are.

    a · b = |a| × |b| × cos θ = 5 × 10 × 1 = 50

Same number, two lenses. Rearranging the second gives **cosine similarity**:

    cos θ = (a · b) / (|a| × |b|)

In [ ]:
print('a . b (by hand) =', (3 * 6) + (4 * 8))
print('a . b (torch)   =', torch.dot(a, b).item())
print('a . c (torch)   =', torch.dot(a, c).item())
print()

cos_ab = torch.dot(a, b) / (torch.linalg.norm(a) * torch.linalg.norm(b))
cos_ac = torch.dot(a, c) / (torch.linalg.norm(a) * torch.linalg.norm(c))
print('cos(a, b) =', cos_ab.item())
print('cos(a, c) =', cos_ac.item())

### Reading the output
- `a · b = 50` — the hand calculation and PyTorch agree.
- `a · c = -50` — **negative**, because `c` points the opposite way. A negative dot product *means* the angle is more than 90°.
- `cos(a, b) = 1.0` — perfectly aligned, the maximum.
- `cos(a, c) = -1.0` — exactly opposite, the minimum.

Cosine is always in `[-1, 1]`, in 2 dimensions or 4096.

## Experiment 3 — the point of the whole lesson

`b` is just `a` doubled. It says nothing new about *direction* — yet `a · b` came out
twice as big as `a · a`, purely because `b` is longer.

So what happens if we make it **100 times** longer?

In [ ]:
b_big = b * 100      # [600, 800] — identical direction, 100x the length

cos_big = torch.dot(a, b_big) / (torch.linalg.norm(a) * torch.linalg.norm(b_big))
print('a . b_big   =', torch.dot(a, b_big).item())
print('cos(a, b_big) =', cos_big.item())

### Reading the output
The dot product explodes from `50` to `5000`. The cosine does not move at all — still `1.0`.

**That is the distinction.** The dot product tracks length; cosine does not. This is why a
search engine ranks by cosine — otherwise a long document would beat a short one just for
being long, which says nothing about whether it answers the query.

## Challenge — predict first, then run

Let `d = [30, 40]` — that is `a` again, ten times longer, same direction.

Predict **both** numbers before running the cell:

- `a · d` = _(write here)_
- `cos(a, d)` = _(write here)_

In [ ]:
# run AFTER writing your prediction above
d = torch.tensor([30.0, 40.0])
print('a . d     =', torch.dot(a, d).item())
print('cos(a, d) =', (torch.dot(a, d) / (torch.linalg.norm(a) * torch.linalg.norm(d))).item())